### Step 1: Setup Environment

In [1]:
!pip install numpy scipy matplotlib torch

In [32]:
!pip install numpy scipy matplotlib torch mlflow

### Step 2: Load Dataset

In [2]:
import json
from pathlib import Path
import numpy as np

DATA_DIR = Path(r"\\PANKAJ-DESKTOP-\Users\JANVI COMPUTER\Downloads\arc-prize-2026-arc-agi-2")

with open(DATA_DIR / "arc-agi_training_challenges.json") as f:
    train_challenges = json.load(f)
with open(DATA_DIR / "arc-agi_training_solutions.json") as f:
    train_solutions = json.load(f)
with open(DATA_DIR / "arc-agi_evaluation_challenges.json") as f:
    eval_challenges = json.load(f)
with open(DATA_DIR / "arc-agi_evaluation_solutions.json") as f:
    eval_solutions = json.load(f)
with open(DATA_DIR / "arc-agi_test_challenges.json") as f:
    test_challenges = json.load(f)

def to_array(g): return np.array(g, dtype=np.uint8)
print(f"Training: {len(train_challenges)} | Eval: {len(eval_challenges)} | Test: {len(test_challenges)}")

Training: 1000 | Eval: 120 | Test: 240


### Step 3: Object Extraction Core (shared by everything below)

In [3]:
from scipy import ndimage

def get_background(grid):
    vals, counts = np.unique(grid, return_counts=True)
    return int(vals[np.argmax(counts)])

def get_objects(grid, connectivity=2, by_color=False):
    bg = get_background(grid)
    objs = []
    if by_color:
        for color in np.unique(grid):
            if color == bg: continue
            mask = grid == color
            structure = ndimage.generate_binary_structure(2, connectivity)
            labeled, n = ndimage.label(mask, structure=structure)
            for i in range(1, n + 1):
                ys, xs = np.where(labeled == i)
                objs.append(_obj_dict(ys, xs, grid))
    else:
        mask = grid != bg
        structure = ndimage.generate_binary_structure(2, connectivity)
        labeled, n = ndimage.label(mask, structure=structure)
        for i in range(1, n + 1):
            ys, xs = np.where(labeled == i)
            objs.append(_obj_dict(ys, xs, grid))
    return objs

def _obj_dict(ys, xs, grid):
    return {
        "ys": ys, "xs": xs,
        "bbox": (int(ys.min()), int(xs.min()), int(ys.max()), int(xs.max())),
        "size": len(ys),
        "colors": set(grid[ys, xs].tolist()),
        "centroid": (float(ys.mean()), float(xs.mean())),
    }

def object_to_grid(obj, grid, bg):
    y0, x0, y1, x1 = obj["bbox"]
    sub = np.full((y1 - y0 + 1, x1 - x0 + 1), bg, dtype=grid.dtype)
    sub[obj["ys"] - y0, obj["xs"] - x0] = grid[obj["ys"], obj["xs"]]
    return sub

def paste(canvas, patch, y0, x0, bg=None, transparent_bg=True):
    canvas = canvas.copy()
    h, w = patch.shape
    ch, cw = canvas.shape
    y1, x1 = min(y0 + h, ch), min(x0 + w, cw)
    if y0 >= ch or x0 >= cw or y1 <= 0 or x1 <= 0:
        return canvas
    py0, px0 = max(0, -y0), max(0, -x0)
    region = patch[py0:py0 + (y1 - max(y0,0)), px0:px0 + (x1 - max(x0,0))]
    if transparent_bg and bg is not None:
        mask = region != bg
        canvas[max(y0,0):y1, max(x0,0):x1][mask] = region[mask]
    else:
        canvas[max(y0,0):y1, max(x0,0):x1] = region
    return canvas

### Expanded DSL: Parametrized Primitive Families
Each function below is genarator that produces many concrete named primitives. This is how hundreds is achieved without hand writting hundreds of one off function

In [4]:
PRIMITIVES = {}

def register(name, fn):
    PRIMITIVES[name] = fn

# ---- 1. Rigid transforms (fixed, ~8) ----
register("identity", lambda g: g)
register("rot90", lambda g: np.rot90(g, 1))
register("rot180", lambda g: np.rot90(g, 2))
register("rot270", lambda g: np.rot90(g, 3))
register("flip_h", np.fliplr)
register("flip_v", np.flipud)
register("transpose", lambda g: g.T)
register("anti_transpose", lambda g: np.fliplr(np.flipud(g.T)))

# ---- 2. Translation family: dx,dy in -5..5 (~121 primitives) ----
def make_translate(dx, dy):
    def fn(g):
        bg = get_background(g)
        canvas = np.full_like(g, bg)
        return paste(canvas, g, dy, dx, bg=bg, transparent_bg=True)
    return fn

for dx in range(-5, 6):
    for dy in range(-5, 6):
        if dx == 0 and dy == 0: continue
        register(f"translate_{dx}_{dy}", make_translate(dx, dy))

# ---- 3. Tiling family: (rows, cols) up to 5x5 (~24 primitives) ----
def make_tile(r, c):
    return lambda g: np.tile(g, (r, c))

for r in range(1, 6):
    for c in range(1, 6):
        if r == 1 and c == 1: continue
        register(f"tile_{r}x{c}", make_tile(r, c))

# ---- 4. Color remap family: swap color A -> color B, for all 10x10 pairs (~90 primitives) ----
def make_recolor(a, b):
    def fn(g):
        out = g.copy()
        out[g == a] = b
        return out
    return fn

for a in range(10):
    for b in range(10):
        if a == b: continue
        register(f"recolor_{a}_to_{b}", make_recolor(a, b))

# ---- 5. Cropping / bounding-box family (~6) ----
def crop_to_content(g):
    bg = get_background(g)
    ys, xs = np.where(g != bg)
    if len(ys) == 0: return g
    return g[ys.min():ys.max()+1, xs.min():xs.max()+1]

def crop_largest_object(g):
    objs = get_objects(g)
    if not objs: return g
    o = max(objs, key=lambda o: o["size"])
    y0, x0, y1, x1 = o["bbox"]
    return g[y0:y1+1, x0:x1+1]

def crop_smallest_object(g):
    objs = get_objects(g)
    if not objs: return g
    o = min(objs, key=lambda o: o["size"])
    y0, x0, y1, x1 = o["bbox"]
    return g[y0:y1+1, x0:x1+1]

def crop_most_common_colored_object(g):
    objs = get_objects(g, by_color=True)
    if not objs: return g
    color_counts = {}
    for o in objs:
        c = next(iter(o["colors"]))
        color_counts[c] = color_counts.get(c, 0) + 1
    target = max(color_counts, key=color_counts.get)
    for o in objs:
        if next(iter(o["colors"])) == target:
            y0, x0, y1, x1 = o["bbox"]
            return g[y0:y1+1, x0:x1+1]
    return g

register("crop_content", crop_to_content)
register("crop_largest_obj", crop_largest_object)
register("crop_smallest_obj", crop_smallest_object)
register("crop_common_color_obj", crop_most_common_colored_object)

# ---- 6. Flood fill family: fill enclosed background regions with each color (~10) ----
def make_flood_fill_holes(fill_color):
    def fn(g):
        bg = get_background(g)
        mask = g == bg
        labeled, n = ndimage.label(mask)
        out = g.copy()
        h, w = g.shape
        for i in range(1, n + 1):
            ys, xs = np.where(labeled == i)
            touches_border = np.any(ys == 0) or np.any(ys == h-1) or np.any(xs == 0) or np.any(xs == w-1)
            if not touches_border:
                out[ys, xs] = fill_color
        return out
    return fn

for color in range(10):
    register(f"flood_fill_holes_{color}", make_flood_fill_holes(color))

# ---- 7. Object-to-object relation ops (~8) ----
def align_objects_to_grid_corner(g, corner="top_left"):
    bg = get_background(g)
    objs = get_objects(g)
    out = np.full_like(g, bg)
    for o in objs:
        patch = object_to_grid(o, g, bg)
        if corner == "top_left":
            out = paste(out, patch, 0, 0, bg=bg)
        elif corner == "top_right":
            out = paste(out, patch, 0, g.shape[1]-patch.shape[1], bg=bg)
    return out

register("align_objs_top_left", lambda g: align_objects_to_grid_corner(g, "top_left"))
register("align_objs_top_right", lambda g: align_objects_to_grid_corner(g, "top_right"))

def sort_objects_by_size_recolor(g):
    """Recolor objects by rank: biggest gets color 1, next color 2, etc."""
    bg = get_background(g)
    objs = sorted(get_objects(g), key=lambda o: -o["size"])
    out = g.copy()
    for rank, o in enumerate(objs, start=1):
        out[o["ys"], o["xs"]] = min(rank, 9)
    return out

register("recolor_by_size_rank", sort_objects_by_size_recolor)

# ---- 8. Symmetry completion (~4) ----
def symmetrize_h(g):
    bg = get_background(g)
    flipped = np.fliplr(g)
    out = g.copy()
    mask = (out == bg) & (flipped != bg)
    out[mask] = flipped[mask]
    return out

def symmetrize_v(g):
    bg = get_background(g)
    flipped = np.flipud(g)
    out = g.copy()
    mask = (out == bg) & (flipped != bg)
    out[mask] = flipped[mask]
    return out

register("symmetrize_h", symmetrize_h)
register("symmetrize_v", symmetrize_v)

# ---- 9. Gravity family (4 directions) ----
def make_gravity(direction):
    def fn(g):
        bg = get_background(g)
        out = np.full_like(g, bg)
        h, w = g.shape
        if direction in ("down", "up"):
            for col in range(w):
                vals = g[:, col][g[:, col] != bg]
                if direction == "down":
                    out[h-len(vals):, col] = vals
                else:
                    out[:len(vals), col] = vals
        else:
            for row in range(h):
                vals = g[row, :][g[row, :] != bg]
                if direction == "right":
                    out[row, w-len(vals):] = vals
                else:
                    out[row, :len(vals)] = vals
        return out
    return fn

for d in ["down", "up", "left", "right"]:
    register(f"gravity_{d}", make_gravity(d))

# ---- 10. Pattern replication: repeat each cell k times (~8) ----
def make_scale(k):
    return lambda g: np.kron(g, np.ones((k, k), dtype=g.dtype))

for k in range(2, 6):
    register(f"scale_{k}x", make_scale(k))

print(f"Total registered primitives: {len(PRIMITIVES)}")

Total registered primitives: 269


This registers on the order of ~280 primitives (121 translations + 90 recolors + 24 tilings + 10 flood-fills + ~35 structural/geometric ones) — genuinely in the "hundreds" range, and every family is trivially extendable (e.g. widen the translate range, add per-color gravity, add per-object recoloring rules) if you want more.

### Step 5: LLM Code Generation

In [5]:
import sys
print(sys.executable)

!"{sys.executable}" -m pip install anthropic

C:\Users\JANVI COMPUTER\anaconda3\envs\tf_env\python.exe


In [6]:
import sys
print(sys.executable)

C:\Users\JANVI COMPUTER\anaconda3\envs\tf_env\python.exe


In [7]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "groq"])

CompletedProcess(args=['C:\\Users\\JANVI COMPUTER\\anaconda3\\envs\\tf_env\\python.exe', '-m', 'pip', 'install', 'groq'], returncode=0)

In [8]:
print(f"Total registered primitives: {len(PRIMITIVES)}")

Total registered primitives: 269


In [9]:
import os
import time
import re
import json
from groq import Groq, RateLimitError

# Supporting multiple API keys for automatic fallback on 429
GROQ_KEYS = [
    "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",
    # Add secondary/backup API keys here if available:
    # "gsk_YourSecondaryKeyHere...",
]

current_key_idx = 0
client = Groq(api_key=GROQ_KEYS[current_key_idx])

CODEGEN_PROMPT = """You are solving an ARC-AGI grid transformation puzzle.
Below are input/output grid pairs showing a transformation rule.

CRITICAL REQUIREMENT: Your response must contain EXACTLY ONE function, named exactly `solve`,
with this exact signature: def solve(grid: np.ndarray) -> np.ndarray:
Do not name it anything else. Do not wrap it in a class. Do not add extra functions.

{examples}

Return ONLY the function code, no markdown fences, no commentary, no explanation.
"""

def format_examples(task):
    parts = []
    for i, pair in enumerate(task["train"]):
        # Compact representation to minimize token overhead
        inp_str = str(pair['input']).replace(' ', '')
        out_str = str(pair['output']).replace(' ', '')
        parts.append(f"Example {i+1} input:\n{inp_str}\nExample {i+1} output:\n{out_str}\n")
    return "\n".join(parts)

def rotate_api_key():
    global current_key_idx, client
    if len(GROQ_KEYS) > 1:
        current_key_idx = (current_key_idx + 1) % len(GROQ_KEYS)
        print(f"  [System] Switching to API Key Index: {current_key_idx}")
        client = Groq(api_key=GROQ_KEYS[current_key_idx])
        return True
    return False

def generate_candidate_solver(task, feedback=None, max_retries=5):
    prompt = CODEGEN_PROMPT.format(examples=format_examples(task))
    if feedback:
        prompt += f"\nYour previous attempt failed with: {feedback}\nFix the function."

    for retry in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000,
                reasoning_effort="low",
            )
            code = resp.choices[0].message.content.strip()
            code = code.replace("```python", "").replace("```", "").strip()
            return code

        except RateLimitError as e:
            err_msg = str(e)
            print(f"  [429 RateLimit] Token/Request Limit Reached.")
            
            # Attempt switching keys first
            if rotate_api_key():
                continue

            # Parse wait time directly from error message (e.g. "try again in 9m47s")
            match = re.search(r"try again in (\d+m)?([\d\.]+)s", err_msg)
            if match:
                minutes = float(match.group(1)[:-1]) if match.group(1) else 0
                seconds = float(match.group(2))
                total_wait = int(minutes * 60 + seconds) + 5
                print(f"  [Auto-Pause] Daily/Minute quota reached. Sleeping for {total_wait} seconds...")
                time.sleep(total_wait)
            else:
                wait = min(2 ** (retry + 2), 120)
                print(f"  [Retry] Retrying in {wait}s... (attempt {retry+1}/{max_retries})")
                time.sleep(wait)

        except Exception as e:
            wait = min(2 ** retry, 30)
            print(f"  API error ({type(e).__name__}): {e}")
            print(f"  Retrying in {wait}s... (attempt {retry+1}/{max_retries})")
            time.sleep(wait)

    return None

### Step 6: Object Extraction Core

In [10]:
import numpy as np
from scipy import ndimage

def get_background(grid):
    vals, counts = np.unique(grid, return_counts=True)
    return int(vals[np.argmax(counts)])

def _obj_dict(ys, xs, grid):
    return {
        "ys": ys, "xs": xs,
        "bbox": (int(ys.min()), int(xs.min()), int(ys.max()), int(xs.max())),
        "size": len(ys),
        "colors": set(grid[ys, xs].tolist()),
        "centroid": (float(ys.mean()), float(xs.mean())),
    }

def get_objects(grid, connectivity=2, by_color=False):
    bg = get_background(grid)
    objs = []
    if by_color:
        for color in np.unique(grid):
            if color == bg:
                continue
            mask = grid == color
            structure = ndimage.generate_binary_structure(2, connectivity)
            labeled, n = ndimage.label(mask, structure=structure)
            for i in range(1, n + 1):
                ys, xs = np.where(labeled == i)
                objs.append(_obj_dict(ys, xs, grid))
    else:
        mask = grid != bg
        structure = ndimage.generate_binary_structure(2, connectivity)
        labeled, n = ndimage.label(mask, structure=structure)
        for i in range(1, n + 1):
            ys, xs = np.where(labeled == i)
            objs.append(_obj_dict(ys, xs, grid))
    return objs

### Step 7: DSL Primitives

In [11]:
PRIMITIVES = {}

def register(name, fn):
    PRIMITIVES[name] = fn

register("identity", lambda g: g)
register("rot90", lambda g: np.rot90(g, 1))
register("rot180", lambda g: np.rot90(g, 2))
register("rot270", lambda g: np.rot90(g, 3))
register("flip_h", np.fliplr)
register("flip_v", np.flipud)
register("transpose", lambda g: g.T)
register("anti_transpose", lambda g: np.fliplr(np.flipud(g.T)))

def make_recolor(a, b):
    def fn(g):
        out = g.copy()
        out[g == a] = b
        return out
    return fn

for a in range(10):
    for b in range(10):
        if a != b:
            register(f"recolor_{a}_to_{b}", make_recolor(a, b))

def make_tile(r, c):
    return lambda g: np.tile(g, (r, c))

for r in range(1, 6):
    for c in range(1, 6):
        if not (r == 1 and c == 1):
            register(f"tile_{r}x{c}", make_tile(r, c))

def crop_to_content(g):
    bg = get_background(g)
    ys, xs = np.where(g != bg)
    if len(ys) == 0:
        return g
    return g[ys.min():ys.max()+1, xs.min():xs.max()+1]

def crop_largest_object(g):
    objs = get_objects(g)
    if not objs:
        return g
    o = max(objs, key=lambda o: o["size"])
    y0, x0, y1, x1 = o["bbox"]
    return g[y0:y1+1, x0:x1+1]

def crop_smallest_object(g):
    objs = get_objects(g)
    if not objs:
        return g
    o = min(objs, key=lambda o: o["size"])
    y0, x0, y1, x1 = o["bbox"]
    return g[y0:y1+1, x0:x1+1]

register("crop_content", crop_to_content)
register("crop_largest_obj", crop_largest_object)
register("crop_smallest_obj", crop_smallest_object)

def symmetrize_h(g):
    bg = get_background(g)
    flipped = np.fliplr(g)
    out = g.copy()
    mask = (out == bg) & (flipped != bg)
    out[mask] = flipped[mask]
    return out

def symmetrize_v(g):
    bg = get_background(g)
    flipped = np.flipud(g)
    out = g.copy()
    mask = (out == bg) & (flipped != bg)
    out[mask] = flipped[mask]
    return out

register("symmetrize_h", symmetrize_h)
register("symmetrize_v", symmetrize_v)

def make_gravity(direction):
    def fn(g):
        bg = get_background(g)
        out = np.full_like(g, bg)
        h, w = g.shape
        if direction in ("down", "up"):
            for col in range(w):
                vals = g[:, col][g[:, col] != bg]
                if direction == "down":
                    out[h-len(vals):, col] = vals
                else:
                    out[:len(vals), col] = vals
        else:
            for row in range(h):
                vals = g[row, :][g[row, :] != bg]
                if direction == "right":
                    out[row, w-len(vals):] = vals
                else:
                    out[row, :len(vals)] = vals
        return out
    return fn

for d in ["down", "up", "left", "right"]:
    register(f"gravity_{d}", make_gravity(d))

def make_scale(k):
    return lambda g: np.kron(g, np.ones((k, k), dtype=g.dtype))

for k in range(2, 6):
    register(f"scale_{k}x", make_scale(k))

print(f"Total registered primitives: {len(PRIMITIVES)}")

Total registered primitives: 135


### Step 8: Chain Application and Scoring

In [12]:
def apply_chain(grid, chain):
    for name in chain:
        try:
            grid = PRIMITIVES[name](grid)
        except Exception:
            return None
        if grid is None or grid.size == 0 or grid.shape[0] > 30 or grid.shape[1] > 30:
            return None
    return grid

def score_chain(chain, train_pairs):
    correct = 0
    for inp, out in train_pairs:
        pred = apply_chain(inp.copy(), chain)
        if pred is not None and pred.shape == out.shape and np.array_equal(pred, out):
            correct += 1
    return correct / len(train_pairs)

### Step 9: Neural Primitive Ranker

In [13]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "torch"])

CompletedProcess(args=['C:\\Users\\JANVI COMPUTER\\anaconda3\\envs\\tf_env\\python.exe', '-m', 'pip', 'install', 'torch'], returncode=0)

In [14]:
import torch
import torch.nn as nn
import numpy as np

def extract_state_features(current, target_shape_hint):
    h, w = current.shape
    n_objs = len(get_objects(current))
    n_colors = len(np.unique(current))
    return np.array([
        h / 30, w / 30,
        n_objs / 10,
        n_colors / 10,
        float(current.shape == target_shape_hint) if target_shape_hint else 0.0,
    ], dtype=np.float32)

class PrimitiveRanker(nn.Module):
    def __init__(self, n_features=5):
        super().__init__()
        self.prim_names = list(PRIMITIVES.keys())
        self.prim_index = {n: i for i, n in enumerate(self.prim_names)}
        self.net = nn.Sequential(
            nn.Linear(n_features + len(self.prim_names), 64), nn.ReLU(),
            nn.Linear(64, 1)
        )

    def score(self, state_feats, prim_name):
        onehot = np.zeros(len(self.prim_names), dtype=np.float32)
        onehot[self.prim_index[prim_name]] = 1.0
        x = torch.tensor(np.concatenate([state_feats, onehot])).unsqueeze(0)
        return self.net(x).item()

ranker = PrimitiveRanker()

### Step 10: Guided Beam Search

In [15]:
def to_array(g):
    return np.array(g, dtype=np.uint8)

def guided_beam_search(task, max_depth=3, beam_width=25, top_k_candidates=40):
    train_pairs = [(to_array(p["input"]), to_array(p["output"])) for p in task["train"]]
    target_shape = train_pairs[0][1].shape if all(o.shape == train_pairs[0][1].shape for _, o in train_pairs) else None

    beam = [([], 0.0)]
    best_chain, best_score = [], 0.0

    for depth in range(max_depth):
        candidates = []
        for chain, _ in beam:
            last_state = train_pairs[0][0] if not chain else apply_chain(train_pairs[0][0].copy(), chain)
            if last_state is None:
                continue
            feats = extract_state_features(last_state, target_shape)
            ranked = sorted(PRIMITIVES.keys(), key=lambda p: ranker.score(feats, p), reverse=True)[:top_k_candidates]

            for prim_name in ranked:
                new_chain = chain + [prim_name]
                s = score_chain(new_chain, train_pairs)
                candidates.append((new_chain, s))
                if s > best_score:
                    best_score, best_chain = s, new_chain
                if s == 1.0:
                    return best_chain, 1.0

        candidates.sort(key=lambda x: x[1], reverse=True)
        beam = candidates[:beam_width]

    return best_chain, best_score

### Step 11: Train the Ranker

In [16]:
def collect_ranker_training_data(tasks, n_tasks=200):
    X, y = [], []
    for tid, task in list(tasks.items())[:n_tasks]:
        train_pairs = [(to_array(p["input"]), to_array(p["output"])) for p in task["train"]]
        target_shape = train_pairs[0][1].shape if all(o.shape == train_pairs[0][1].shape for _, o in train_pairs) else None
        for prim_name in PRIMITIVES:
            s = score_chain([prim_name], train_pairs)
            feats = extract_state_features(train_pairs[0][0], target_shape)
            X.append((feats, prim_name))
            y.append(1.0 if s > 0.0 else 0.0)
    return X, y

X, y = collect_ranker_training_data(train_challenges, n_tasks=200)

opt = torch.optim.Adam(ranker.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(20):
    total_loss = 0
    for (feats, prim_name), label in zip(X, y):
        onehot = np.zeros(len(ranker.prim_names), dtype=np.float32)
        onehot[ranker.prim_index[prim_name]] = 1.0
        x = torch.tensor(np.concatenate([feats, onehot])).unsqueeze(0)
        target = torch.tensor([[label]])
        pred = ranker.net(x)
        loss = loss_fn(pred, target)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"epoch {epoch} loss {total_loss/len(X):.4f}")

epoch 0 loss 0.0380
epoch 1 loss 0.0233
epoch 2 loss 0.0227
epoch 3 loss 0.0232
epoch 4 loss 0.0203
epoch 5 loss 0.0180
epoch 6 loss 0.0167
epoch 7 loss 0.0160
epoch 8 loss 0.0159
epoch 9 loss 0.0162
epoch 10 loss 0.0159
epoch 11 loss 0.0167
epoch 12 loss 0.0161
epoch 13 loss 0.0156
epoch 14 loss 0.0156
epoch 15 loss 0.0163
epoch 16 loss 0.0172
epoch 17 loss 0.0165
epoch 18 loss 0.0163
epoch 19 loss 0.0160


In [33]:
import os
import urllib.parse
from pathlib import Path
import torch
import mlflow
import mlflow.pyfunc
import numpy as np

# 1. Allow local file store in MLflow (or use SQLite backend)
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

# Fix working directory encoding if %20 is present in path
current_dir = os.getcwd()
if "%20" in current_dir:
    clean_dir = urllib.parse.unquote(current_dir)
    os.chdir(clean_dir)

# Option A: SQLite backend (Recommended by MLflow)
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Option B: Local file backend (Works because MLFLOW_ALLOW_FILE_STORE="true")
# local_mlruns_path = Path('.').resolve().as_posix()
# mlflow.set_tracking_uri(f"file:///{local_mlruns_path}/mlruns")


# ==========================================
# 1. DEFINE MLFLOW WRAPPER CLASS
# ==========================================
class ARCSolverMLflowWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        """Loads model weights and reconstructs the pipeline state."""
        self.ranker = PrimitiveRanker()
        self.ranker.load_state_dict(torch.load(context.artifacts["ranker_weights"]))
        self.ranker.eval()

    def predict(self, context, model_input: list[dict | list]) -> list:
        """
        model_input: batch list containing task dictionaries or items:
          - 'task': task dict with 'train' and 'test' keys
          - 'max_depth': optional beam search depth (default=3)
        """
        predictions = []
        if isinstance(model_input, list):
            tasks = model_input
        else:
            tasks = [model_input]

        for item in tasks:
            task = item.get("task")
            max_depth = item.get("max_depth", 3)
            
            # Execute guided beam search using saved ranker
            chain, score = guided_beam_search(task, max_depth=max_depth)
            
            if score == 1.0:
                preds = [apply_chain(to_array(t["input"]), chain) for t in task["test"]]
                predictions.append({
                    "status": "solved_by_dsl", 
                    "chain": chain, 
                    "predictions": [p.tolist() for p in preds]
                })
            else:
                predictions.append({
                    "status": "unsolved_by_dsl", 
                    "chain": chain, 
                    "score": score
                })
                
        return predictions


# ==========================================
# 2. LOG AND REGISTER MODEL TO MLFLOW
# ==========================================
mlflow.set_experiment("ARC_AGI_2026_Hybrid_Solver")

with mlflow.start_run(run_name="Ranker_and_BeamSearch_Pipeline") as run:
    # --- A. Save Neural Ranker Weights ---
    weights_path = "ranker_weights.pth"
    torch.save(ranker.state_dict(), weights_path)

    # --- B. Log Hyperparameters & Metrics ---
    mlflow.log_param("num_primitives", len(PRIMITIVES))
    mlflow.log_param("beam_width", 25)
    mlflow.log_param("max_depth", 3)
    mlflow.log_metric("dsl_accuracy_pct", 1.50)

    # --- C. Define Artifacts & Log Custom PyFunc Model ---
    artifacts = {
        "ranker_weights": os.path.abspath(weights_path)
    }

    conda_env = {
        'channels': ['defaults', 'conda-forge'],
        'dependencies': [
            'python=3.11',
            'pip',
            {
                'pip': [
                    'mlflow',
                    'torch',
                    'numpy',
                    'scipy',
                    'groq'
                ],
            },
        ],
        'name': 'arc_solver_env'
    }

    model_info = mlflow.pyfunc.log_model(
        artifact_path="arc_hybrid_solver",
        python_model=ARCSolverMLflowWrapper(),
        artifacts=artifacts,
        conda_env=conda_env,
        registered_model_name="ARC_Hybrid_Solver"
    )

    print(f"Model successfully logged at: {model_info.model_uri}")

2026/09/28 14:49:11 WARNING mlflow.types.type_hints: Union type hint is inferred as AnyType, and MLflow doesn't validate the data against its element types.
2026/09/28 14:49:13 INFO mlflow.tracking.fluent: Experiment with name 'ARC_AGI_2026_Hybrid_Solver' does not exist. Creating a new experiment.
2026/09/28 14:49:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/28 14:49:20 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/09/28 14:49:20 WARNING mlflow.types.type_hints: Union type hint is inferred as AnyType, and MLflow doesn't validate the data against its element types.
2026/09/28 14:49:20 INFO mlflow.models.signature: Failed t

Model successfully logged at: models:/m-e0f88cd3c883479ca97cf2b25d5cebc0


In [46]:
import mlflow.pyfunc
import numpy as np

# 1. Load registered model from MLflow registry
model_uri = "models:/ARC_Hybrid_Solver/1"
loaded_solver = mlflow.pyfunc.load_model(model_uri)

# 2. Programmatically define a valid ARC task structure
synthetic_arc_task = {
    "train": [
        {
            "input": [[1, 0, 0], [0, 1, 0], [0, 0, 1]],
            "output": [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
        },
        {
            "input": [[2, 2], [2, 2]],
            "output": [[2, 2], [2, 2]]
        }
    ],
    "test": [
        {
            "input": [[3, 0], [0, 3]]
        }
    ]
}

task_id = "Synthetic_Test_Task_01"

# 3. Format input payload for MLflow predict
sample_input = [{
    "task": synthetic_arc_task,
    "max_depth": 3
}]

# 4. Execute prediction
results = loaded_solver.predict(sample_input)

print("\n--- Real Task Prediction Output ---")
print(f"Task ID: {task_id}")
print(f"Status: {results[0]['status']}")

if results[0]['status'] == 'solved_by_dsl':
    print(f"Discovered DSL Chain: {results[0]['chain']}")
    print(f"Predicted Output Grid Shape: {np.array(results[0]['predictions'][0]).shape}")
else:
    print(f"Best DSL Match Score: {results[0]['score']}")


--- Real Task Prediction Output ---
Task ID: Synthetic_Test_Task_01
Status: solved_by_dsl
Discovered DSL Chain: ['crop_largest_obj']
Predicted Output Grid Shape: (2, 2)


In [38]:
# Check which dataset variables are available in memory
for var_name in ['train_tasks', 'tasks', 'arc_tasks', 'data', 'train_data']:
    if var_name in globals():
        print(f"Found dataset variable: {var_name}")

### Step 12: Check DSL-Only Accuracy, Identify Tasks Needing LLM Fallback

In [17]:
dsl_solved, dsl_total = 0, 0
unsolved_tids = []

for tid, task in train_challenges.items():
    chain, score = guided_beam_search(task)
    truths = train_solutions[tid]
    dsl_total += 1
    if score == 1.0:
        preds = [apply_chain(to_array(t["input"]), chain) for t in task["test"]]
        all_correct = all(np.array_equal(preds[i], to_array(truths[i])) for i in range(len(truths)))
        if all_correct:
            dsl_solved += 1
        else:
            unsolved_tids.append(tid)
    else:
        unsolved_tids.append(tid)

print(f"DSL-only: {dsl_solved}/{dsl_total} solved ({100*dsl_solved/dsl_total:.2f}%)")
print(f"{len(unsolved_tids)} tasks remain for LLM fallback")

DSL-only: 15/1000 solved (1.50%)
985 tasks remain for LLM fallback


In [49]:
%%writefile arc_solver_lib.py
"""Standalone copy of the ARC DSL + Ranker + Beam Search."""
import numpy as np
import torch
import torch.nn as nn
from scipy import ndimage

# ---------------- Object extraction ----------------
def get_background(grid):
    vals, counts = np.unique(grid, return_counts=True)
    return int(vals[np.argmax(counts)])

def _obj_dict(ys, xs, grid):
    return {
        "ys": ys, "xs": xs,
        "bbox": (int(ys.min()), int(xs.min()), int(ys.max()), int(xs.max())),
        "size": len(ys),
        "colors": set(grid[ys, xs].tolist()),
        "centroid": (float(ys.mean()), float(xs.mean())),
    }

def get_objects(grid, connectivity=2, by_color=False):
    bg = get_background(grid)
    objs = []
    if by_color:
        for color in np.unique(grid):
            if color == bg:
                continue
            mask = grid == color
            structure = ndimage.generate_binary_structure(2, connectivity)
            labeled, n = ndimage.label(mask, structure=structure)
            for i in range(1, n + 1):
                ys, xs = np.where(labeled == i)
                objs.append(_obj_dict(ys, xs, grid))
    else:
        mask = grid != bg
        structure = ndimage.generate_binary_structure(2, connectivity)
        labeled, n = ndimage.label(mask, structure=structure)
        for i in range(1, n + 1):
            ys, xs = np.where(labeled == i)
            objs.append(_obj_dict(ys, xs, grid))
    return objs

# ---------------- DSL primitives (the 135-primitive version) ----------------
PRIMITIVES = {}

def register(name, fn):
    PRIMITIVES[name] = fn

register("identity", lambda g: g)
register("rot90", lambda g: np.rot90(g, 1))
register("rot180", lambda g: np.rot90(g, 2))
register("rot270", lambda g: np.rot90(g, 3))
register("flip_h", np.fliplr)
register("flip_v", np.flipud)
register("transpose", lambda g: g.T)
register("anti_transpose", lambda g: np.fliplr(np.flipud(g.T)))

def make_recolor(a, b):
    def fn(g):
        out = g.copy()
        out[g == a] = b
        return out
    return fn

for a in range(10):
    for b in range(10):
        if a != b:
            register(f"recolor_{a}_to_{b}", make_recolor(a, b))

def make_tile(r, c):
    return lambda g: np.tile(g, (r, c))

for r in range(1, 6):
    for c in range(1, 6):
        if not (r == 1 and c == 1):
            register(f"tile_{r}x{c}", make_tile(r, c))

def crop_to_content(g):
    bg = get_background(g)
    ys, xs = np.where(g != bg)
    if len(ys) == 0:
        return g
    return g[ys.min():ys.max()+1, xs.min():xs.max()+1]

def crop_largest_object(g):
    objs = get_objects(g)
    if not objs:
        return g
    o = max(objs, key=lambda o: o["size"])
    y0, x0, y1, x1 = o["bbox"]
    return g[y0:y1+1, x0:x1+1]

def crop_smallest_object(g):
    objs = get_objects(g)
    if not objs:
        return g
    o = min(objs, key=lambda o: o["size"])
    y0, x0, y1, x1 = o["bbox"]
    return g[y0:y1+1, x0:x1+1]

register("crop_content", crop_to_content)
register("crop_largest_obj", crop_largest_object)
register("crop_smallest_obj", crop_smallest_object)

def symmetrize_h(g):
    bg = get_background(g)
    flipped = np.fliplr(g)
    out = g.copy()
    mask = (out == bg) & (flipped != bg)
    out[mask] = flipped[mask]
    return out

def symmetrize_v(g):
    bg = get_background(g)
    flipped = np.flipud(g)
    out = g.copy()
    mask = (out == bg) & (flipped != bg)
    out[mask] = flipped[mask]
    return out

register("symmetrize_h", symmetrize_h)
register("symmetrize_v", symmetrize_v)

def make_gravity(direction):
    def fn(g):
        bg = get_background(g)
        out = np.full_like(g, bg)
        h, w = g.shape
        if direction in ("down", "up"):
            for col in range(w):
                vals = g[:, col][g[:, col] != bg]
                if direction == "down":
                    out[h-len(vals):, col] = vals
                else:
                    out[:len(vals), col] = vals
        else:
            for row in range(h):
                vals = g[row, :][g[row, :] != bg]
                if direction == "right":
                    out[row, w-len(vals):] = vals
                else:
                    out[row, :len(vals)] = vals
        return out
    return fn

for d in ["down", "up", "left", "right"]:
    register(f"gravity_{d}", make_gravity(d))

def make_scale(k):
    return lambda g: np.kron(g, np.ones((k, k), dtype=g.dtype))

for k in range(2, 6):
    register(f"scale_{k}x", make_scale(k))

# ---------------- Chain application & scoring ----------------
def to_array(g):
    return np.array(g, dtype=np.uint8)

def apply_chain(grid, chain):
    for name in chain:
        try:
            grid = PRIMITIVES[name](grid)
        except Exception:
            return None
        if grid is None or grid.size == 0 or grid.shape[0] > 30 or grid.shape[1] > 30:
            return None
    return grid

def score_chain(chain, train_pairs):
    correct = 0
    for inp, out in train_pairs:
        pred = apply_chain(inp.copy(), chain)
        if pred is not None and pred.shape == out.shape and np.array_equal(pred, out):
            correct += 1
    return correct / len(train_pairs)

# ---------------- Neural ranker ----------------
def extract_state_features(current, target_shape_hint):
    h, w = current.shape
    n_objs = len(get_objects(current))
    n_colors = len(np.unique(current))
    return np.array([
        h / 30, w / 30,
        n_objs / 10,
        n_colors / 10,
        float(current.shape == target_shape_hint) if target_shape_hint else 0.0,
    ], dtype=np.float32)

class PrimitiveRanker(nn.Module):
    def __init__(self, n_features=5):
        super().__init__()
        self.prim_names = list(PRIMITIVES.keys())
        self.prim_index = {n: i for i, n in enumerate(self.prim_names)}
        self.net = nn.Sequential(
            nn.Linear(n_features + len(self.prim_names), 64), nn.ReLU(),
            nn.Linear(64, 1)
        )

    def score(self, state_feats, prim_name):
        onehot = np.zeros(len(self.prim_names), dtype=np.float32)
        onehot[self.prim_index[prim_name]] = 1.0
        x = torch.tensor(np.concatenate([state_feats, onehot])).unsqueeze(0)
        with torch.no_grad():
            return self.net(x).item()

    def rank(self, state_feats, top_k):
        """Same ordering as sorting by score(), but scores all primitives in one batch (faster)."""
        n = len(self.prim_names)
        x = np.concatenate([np.tile(state_feats, (n, 1)), np.eye(n, dtype=np.float32)], axis=1)
        with torch.no_grad():
            s = self.net(torch.tensor(x)).squeeze(1).tolist()
        order = sorted(range(n), key=lambda i: s[i], reverse=True)[:top_k]
        return [self.prim_names[i] for i in order]

# ---------------- Guided beam search (ranker is now an explicit argument) ----------------
def guided_beam_search(task, ranker, max_depth=3, beam_width=25, top_k_candidates=40):
    train_pairs = [(to_array(p["input"]), to_array(p["output"])) for p in task["train"]]
    target_shape = train_pairs[0][1].shape if all(o.shape == train_pairs[0][1].shape for _, o in train_pairs) else None

    beam = [([], 0.0)]
    best_chain, best_score = [], 0.0

    for depth in range(max_depth):
        candidates = []
        for chain, _ in beam:
            last_state = train_pairs[0][0] if not chain else apply_chain(train_pairs[0][0].copy(), chain)
            if last_state is None:
                continue
            feats = extract_state_features(last_state, target_shape)
            ranked = ranker.rank(feats, top_k_candidates)
            for prim_name in ranked:
                new_chain = chain + [prim_name]
                s = score_chain(new_chain, train_pairs)
                candidates.append((new_chain, s))
                if s > best_score:
                    best_score, best_chain = s, new_chain
                if s == 1.0:
                    return best_chain, 1.0
        candidates.sort(key=lambda x: x[1], reverse=True)
        beam = candidates[:beam_width]
    return best_chain, best_score

# ---------------- Optional LLM-code fallback ----------------
def safe_exec_solver(code_str, grid):
    local_ns = {"np": np, "ndimage": ndimage}
    try:
        exec(code_str, local_ns)
        solve_fn = local_ns.get("solve")
        if solve_fn is None:
            return None
        return solve_fn(grid.copy())
    except Exception:
        return None

Writing arc_solver_lib.py


In [50]:
%%writefile arc_mlflow_model.py
import json
import mlflow
import numpy as np
import pandas as pd
import torch
from mlflow.pyfunc import PythonModel

import arc_solver_lib as L


def _clean(v, default=None):
    if v is None:
        return default
    if isinstance(v, float) and np.isnan(v):
        return default
    return v


class ARCSolverMLflowWrapper(PythonModel):
    def load_context(self, context):
        self.ranker = L.PrimitiveRanker()
        state = torch.load(context.artifacts["ranker_weights"], map_location="cpu")
        self.ranker.load_state_dict(state)
        self.ranker.eval()

        self.llm_solvers = {}
        if "llm_solvers" in context.artifacts:
            with open(context.artifacts["llm_solvers"]) as f:
                self.llm_solvers = json.load(f)

    @staticmethod
    def _rows(model_input):
        if isinstance(model_input, pd.DataFrame):
            return model_input.to_dict(orient="records")
        if isinstance(model_input, dict):
            return [model_input]
        if isinstance(model_input, (list, tuple)):
            return list(model_input)
        return [{"task_json": model_input}]

    def predict(self, context, model_input, params=None):
        results = []
        for row in self._rows(model_input):
            if isinstance(row, str):
                row = {"task_json": row}
            task = _clean(row.get("task_json"), None)
            if task is None:
                task = row.get("task")
            if isinstance(task, str):
                task = json.loads(task)
            max_depth = int(_clean(row.get("max_depth"), 3))
            task_id = _clean(row.get("task_id"), None)

            chain, score = L.guided_beam_search(task, self.ranker, max_depth=max_depth)

            if score == 1.0:
                preds = [L.apply_chain(L.to_array(t["input"]), chain) for t in task["test"]]
                if all(p is not None for p in preds):
                    results.append({"status": "solved_by_dsl", "chain": chain, "score": float(score),
                                    "predictions": [p.tolist() for p in preds]})
                    continue

            code = self.llm_solvers.get(task_id) if task_id else None
            if code:
                preds = [L.safe_exec_solver(code, L.to_array(t["input"])) for t in task["test"]]
                if all(p is not None for p in preds):
                    results.append({"status": "solved_by_llm_code", "chain": chain, "score": float(score),
                                    "predictions": [np.asarray(p).tolist() for p in preds]})
                    continue

            results.append({"status": "unsolved", "chain": chain, "score": float(score), "predictions": None})
        return results


mlflow.models.set_model(ARCSolverMLflowWrapper())

Writing arc_mlflow_model.py


In [51]:
import os, json, importlib
from importlib.metadata import version
import torch, mlflow
import arc_solver_lib
importlib.reload(arc_solver_lib)

# Safety check: packaged primitives must match the ones your ranker was trained with
assert list(arc_solver_lib.PRIMITIVES.keys()) == ranker.prim_names, \
    "Primitive list mismatch - re-run Step 7 and Step 9-11 before this cell."
print("OK - primitives match:", len(ranker.prim_names))

torch.save(ranker.state_dict(), "ranker_weights.pth")

artifacts = {"ranker_weights": "ranker_weights.pth"}
if os.path.exists("llm_generated_solvers.json"):
    artifacts["llm_solvers"] = "llm_generated_solvers.json"

dsl_acc = 100 * dsl_solved / dsl_total if "dsl_solved" in globals() else 1.50
pip_reqs = [f"{p}=={version(p)}" for p in ["mlflow", "torch", "numpy", "scipy", "pandas"]]

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("ARC_AGI_2026_Hybrid_Solver")

with mlflow.start_run(run_name="Ranker_and_BeamSearch_Pipeline"):
    mlflow.log_param("num_primitives", len(ranker.prim_names))
    mlflow.log_param("beam_width", 25)
    mlflow.log_param("top_k_candidates", 40)
    mlflow.log_param("max_depth", 3)
    mlflow.log_param("ranker_train_tasks", 200)
    mlflow.log_param("ranker_epochs", 20)
    mlflow.log_metric("dsl_accuracy_pct_train", dsl_acc)
    mlflow.log_metric("combined_dsl_llm_accuracy_pct_train", 1.77)  # from Step 15

    kwargs = dict(
        python_model="arc_mlflow_model.py",
        artifacts=artifacts,
        code_paths=["arc_solver_lib.py"],
        pip_requirements=pip_reqs,
        registered_model_name="ARC_Hybrid_Solver",
    )
    try:
        model_info = mlflow.pyfunc.log_model(name="arc_hybrid_solver", **kwargs)          # MLflow 3.x
    except TypeError:
        model_info = mlflow.pyfunc.log_model(artifact_path="arc_hybrid_solver", **kwargs)  # MLflow 2.x

print("Model logged at:", model_info.model_uri)

OK - primitives match: 135


C:\Users\JANVI COMPUTER\anaconda3\envs\tf_env\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


Model logged at: models:/m-21572e390e2647458266e6e6a3e19235


Registered model 'ARC_Hybrid_Solver' already exists. Creating a new version of this model...
Created version '2' of model 'ARC_Hybrid_Solver'.


In [52]:
loaded = mlflow.pyfunc.load_model(model_info.model_uri)

tid = list(train_challenges.keys())[0]
out = loaded.predict([{"task_json": json.dumps(train_challenges[tid]), "task_id": tid}])
print(out)

C:\Users\JANVI COMPUTER\anaconda3\envs\tf_env\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


[{'status': 'solved_by_llm_code', 'chain': [], 'score': 0.0, 'predictions': [[[3, 2, 3, 2, 3, 2], [7, 8, 7, 8, 7, 8], [2, 3, 2, 3, 2, 3], [8, 7, 8, 7, 8, 7], [3, 2, 3, 2, 3, 2], [7, 8, 7, 8, 7, 8]]]}]


### Step 13: LLM Fallback (Checkpointed, Retry-Protected, Only on Unsolved Tasks)

In [18]:
def safe_exec_solver(code_str, grid, timeout_sec=3):
    local_ns = {"np": np, "ndimage": ndimage}
    try:
        exec(code_str, local_ns)

        # Find the function regardless of its name
        solve_fn = local_ns.get("solve")
        if solve_fn is None:
            # fallback: grab the first user-defined function in the namespace
            candidates = [v for k, v in local_ns.items()
                          if callable(v) and k not in ("np", "ndimage")]
            if candidates:
                solve_fn = candidates[0]
            else:
                print("    safe_exec_solver failed: no callable function found")
                return None

        return solve_fn(grid.copy())
    except Exception as e:
        print(f"    safe_exec_solver failed: {type(e).__name__}: {e}")
        return None

In [19]:
models = client.models.list()
for m in models.data:
    print(m.id)

whisper-large-v3
allam-2-7b
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
openai/gpt-oss-120b
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-22m


In [20]:
test_tid = list(train_challenges.keys())[0]
task = train_challenges[test_tid]
raw_code = generate_candidate_solver(task)
print(repr(raw_code))
print("---LENGTH---", len(raw_code) if raw_code else "None")

'import numpy as np\n\ndef solve(grid: np.ndarray) -> np.ndarray:\n    base = grid\n    trans = base.T\n    out = np.zeros((6, 6), dtype=grid.dtype)\n    for bi in range(3):\n        for bj in range(3):\n            block = base if (bi + bj) % 2 == 0 else trans\n            out[bi*2:(bi+1)*2, bj*2:(bj+1)*2] = block\n    return out'
---LENGTH--- 321


In [21]:
train_pairs = [(to_array(p["input"]), to_array(p["output"])) for p in task["train"]]
for inp, out in train_pairs:
    pred = safe_exec_solver(raw_code, inp)
    match = pred is not None and pred.shape == out.shape and np.array_equal(pred, out)
    print("Match:", match)

Match: False
Match: False


In [24]:
import re
import time
from groq import Groq, RateLimitError

# List your Groq keys for automatic fallback
GROQ_KEYS = [
    os.environ.get("GROQ_API_KEY"),
    # Add extra key strings here if available:
    # "gsk_...",
]
current_key_idx = 0

def rotate_api_key():
    global current_key_idx, client
    valid_keys = [k for k in GROQ_KEYS if k]
    if len(valid_keys) > 1:
        current_key_idx = (current_key_idx + 1) % len(valid_keys)
        print(f"  [System] Rotating to API Key #{current_key_idx + 1}")
        client = Groq(api_key=valid_keys[current_key_idx])
        return True
    return False

def parse_wait_time(error_message):
    """Extracts wait time in seconds from Groq 429 error messages."""
    match = re.search(r"try again in (?:(\d+)h)?(?:(\d+)m)?([\d\.]+)s", error_message)
    if match:
        hours = float(match.group(1)) if match.group(1) else 0
        minutes = float(match.group(2)) if match.group(2) else 0
        seconds = float(match.group(3)) if match.group(3) else 0
        return int(hours * 3600 + minutes * 60 + seconds) + 5
    return 60

def generate_candidate_solver(task, feedback=None, max_retries=5):
    prompt = CODEGEN_PROMPT.format(examples=format_examples(task))
    if feedback:
        prompt += f"\nYour previous attempt failed with: {feedback}\nFix the function."

    for retry in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000,
                reasoning_effort="low",
            )
            code = resp.choices[0].message.content.strip()
            code = code.replace("```python", "").replace("```", "").strip()
            return code

        except RateLimitError as e:
            err_msg = str(e)
            print(f"  [429 Rate Limit] Quota exceeded.")

            # Try key rotation first
            if rotate_api_key():
                continue

            # If no alternative key, sleep for the requested reset window
            wait_seconds = parse_wait_time(err_msg)
            print(f"  [Quota Reset] Pausing execution for {wait_seconds // 60}m {wait_seconds % 60}s...")
            time.sleep(wait_seconds)

        except Exception as e:
            wait = min(2 ** retry, 60)
            print(f"  API error ({type(e).__name__}): {e}")
            print(f"  Retrying in {wait}s... (attempt {retry+1}/{max_retries})")
            time.sleep(wait)

    return None

### Step 14: Combined Solver

In [25]:
def solve_task_combined(task, tid=None, llm_solvers=None):
    chain, score = guided_beam_search(task)
    if score == 1.0:
        return [apply_chain(to_array(t["input"]), chain) for t in task["test"]]

    if llm_solvers and tid in llm_solvers:
        code = llm_solvers[tid]
        preds = []
        for t in task["test"]:
            pred = safe_exec_solver(code, to_array(t["input"]))
            if pred is None:
                return None
            preds.append(pred)
        return preds

    return None

### Step 15: Full Accuracy Check (DSL + LLM Combined)

In [26]:
with open("llm_generated_solvers.json") as f:
    llm_solvers = json.load(f)

solved, total = 0, 0
for tid, task in train_challenges.items():
    preds = solve_task_combined(task, tid=tid, llm_solvers=llm_solvers)
    truths = train_solutions[tid]
    for i, truth in enumerate(truths):
        total += 1
        if preds is not None and np.array_equal(preds[i], to_array(truth)):
            solved += 1

print(f"Combined solver: {solved}/{total} solved ({100*solved/total:.2f}%)")

Combined solver: 19/1076 solved (1.77%)


### Step 16: Build Submission

In [27]:
def make_submission(challenges, llm_solvers=None):
    submission = {}
    for tid, task in challenges.items():
        preds = solve_task_combined(task, tid=tid, llm_solvers=llm_solvers)
        entries = []
        for i, test_pair in enumerate(task["test"]):
            attempt = preds[i].tolist() if preds is not None else test_pair["input"]
            entries.append({"attempt_1": attempt, "attempt_2": attempt})
        submission[tid] = entries
    return submission

final_submission = make_submission(test_challenges, llm_solvers=llm_solvers)
with open("submission.json", "w") as f:
    json.dump(final_submission, f)

print("Saved submission.json with", len(final_submission), "task entries")

Saved submission.json with 240 task entries


### Step 17: Validate Format

In [28]:
missing = set(test_challenges.keys()) - set(final_submission.keys())
assert not missing, f"Missing task_ids: {missing}"

for tid, entries in final_submission.items():
    for e in entries:
        assert "attempt_1" in e and "attempt_2" in e, f"Missing attempts in {tid}"

for tid, task in test_challenges.items():
    assert len(final_submission[tid]) == len(task["test"]), f"Output count mismatch: {tid}"

print("Submission structure passes all format checks.")

Submission structure passes all format checks.
